# 04 Train Final Cross Validation

Current run: fold 1, seeds 42/43/44; this does not complete all five folds.
Inputs: hcsinhgoethe/ppg-cvae-prepared-fold01 and inputs/final/fold_01 supplied through KRun --input.
Configuration selected by notebook 03: beta=0.01, lambda_hr=0.1.
Expected outputs: artifacts/cvae/fold_01/seed_42, seed_43, seed_44.
Validation HR accuracy is currently low. This run assesses seed stability; test stays closed.


In [ ]:
from pathlib import Path
import os, sys, platform, re, torch

print("=== CHECKING /kaggle/input ===")
input_dir = Path("/kaggle/input")
if input_dir.exists():
    items = list(input_dir.iterdir())
    print(f"Contents of /kaggle/input ({len(items)} items): {[p.name for p in items]}")
    for it in items:
        if it.is_dir():
            print(f"  Folder {it.name} contains {len(list(it.iterdir()))} items")
else:
    print("/kaggle/input does not exist!")

# Robust discovery of DATA_ROOT
DATA_ROOT = None
if os.environ.get('PPGDALIA_ROOT'):
    p = Path(os.environ['PPGDALIA_ROOT'])
    if p.exists():
        DATA_ROOT = p

if DATA_ROOT is None and input_dir.exists():
    for p in input_dir.rglob("*.pkl"):
        if re.search(r'(?<![A-Z0-9])S(1[0-5]|[1-9])(?![A-Z0-9])', p.stem.upper()):
            rel = p.relative_to(input_dir)
            DATA_ROOT = input_dir / rel.parts[0]
            print(f"Auto-discovered DATA_ROOT via subject files: {DATA_ROOT}")
            break

if DATA_ROOT is None:
    _candidates = [
        Path('/kaggle/input/ppg-dalia-dataset'),
        Path('/kaggle/input/ppg-dalia'),
        Path('/kaggle/input/datasets/ameersifat53/ppg-dalia-dataset')
    ]
    DATA_ROOT = next((p for p in _candidates if p.exists() and p.is_dir()), _candidates[0])

# Project-relative output is collected by krun.yaml outputs: artifacts/.
OUTPUT_ROOT = Path(os.environ.get('PPGCVAE_OUTPUT_ROOT', 'artifacts'))
FOLD = 1
SEEDS = (42, 43, 44)
print({'DATA_ROOT': str(DATA_ROOT), 'OUTPUT_ROOT': str(OUTPUT_ROOT), 'device': 'cuda' if torch.cuda.is_available() else 'cpu', 'python': sys.version, 'torch': torch.__version__})
# This stage consumes prepared arrays/checkpoints; DATA_ROOT is not required.


In [ ]:
# Locate this repository without assuming a Kaggle dataset slug.
repo_candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'Final_IOT_for_AI']
repo_root = next((p for p in repo_candidates if (p / 'src').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate repository src/ directory; upload/clone the repository first.')
sys.path.insert(0, str(repo_root / 'src'))
from ppg_cvae.data.splits import get_fold
print(get_fold(FOLD))

# Prepared dataset: https://www.kaggle.com/datasets/hcsinhgoethe/ppg-cvae-prepared-fold01
import shutil
from zipfile import ZipFile
DATASET_SLUG = 'ppg-cvae-prepared-fold01'
ARTIFACT_INPUT_ROOT = os.environ.get('PPGCVAE_INPUT_ROOT')
fold_name = f'fold_{FOLD:02d}'
required_files = ('train.npz', 'val.npz', 'manifest.csv', 'normalization_stats.json', 'preprocessing_config.json')
destination = OUTPUT_ROOT/'prepared'/fold_name

# Discover the mounted dataset rather than assuming Kaggle's directory layout.
if ARTIFACT_INPUT_ROOT:
    dataset_roots = [Path(ARTIFACT_INPUT_ROOT)]
else:
    dataset_roots = sorted(p for p in input_dir.rglob(DATASET_SLUG) if p.is_dir()) if input_dir.exists() else []
if not dataset_roots:
    raise FileNotFoundError(f'Attach hcsinhgoethe/{DATASET_SLUG} with krun --dataset, or set PPGCVAE_INPUT_ROOT.')

prepared_sources = []
for root in dataset_roots:
    if not root.is_dir(): raise FileNotFoundError(root)
    for train_file in root.rglob('train.npz'):
        candidate = train_file.parent
        if candidate.name.startswith('fold_') and candidate.name != fold_name: continue
        if all((candidate/name).is_file() for name in required_files):
            prepared_sources.append(candidate)
if len(prepared_sources) > 1:
    raise ValueError(f'Ambiguous prepared artifact directories: {prepared_sources}; set PPGCVAE_INPUT_ROOT explicitly.')
if prepared_sources:
    prepared_source = prepared_sources[0]
    if prepared_source.resolve() != destination.resolve():
        shutil.copytree(prepared_source, destination, dirs_exist_ok=True)
    print('Imported prepared arrays from:', prepared_source)
else:
    # Also support a dataset whose uploaded ZIP has not been unpacked by Kaggle.
    archives = []
    for root in dataset_roots:
        for archive in root.rglob('*.zip'):
            with ZipFile(archive) as z:
                prefix = f'prepared/{fold_name}/'
                if all(prefix+name in z.namelist() for name in required_files):
                    archives.append(archive)
    if len(archives) != 1:
        raise FileNotFoundError(f'Expected one prepared {fold_name} directory or ZIP in {dataset_roots}; found archives: {archives}')
    destination.mkdir(parents=True, exist_ok=True)
    with ZipFile(archives[0]) as z:
        prefix = f'prepared/{fold_name}/'
        for name in (*required_files, 'test.npz', 'subject_summary.csv', 'hr_coverage.csv'):
            member = prefix+name
            if member in z.namelist():
                with z.open(member) as src, (destination/name).open('wb') as dst:
                    shutil.copyfileobj(src, dst)
    print('Imported prepared arrays from ZIP:', archives[0])
assert all((destination/name).is_file() for name in required_files)
print('Prepared fold directory:', destination)

# Proxy is a differentiable estimate, not a ground-truth HR label.
# Preserve validation quality metadata in every tuning run for interpretation.
final_input = repo_root/'inputs'/'final'/fold_name
proxy_source = final_input/'proxy'
for name in ('hr_proxy.pt','detector_config.json','hr_proxy_validation_metrics.json','detector_validation_metrics.json'):
    if not (proxy_source/name).is_file():
        raise FileNotFoundError(f'Missing {proxy_source/name}; pass --input inputs/final/fold_01 to KRun.')
proxy_destination = OUTPUT_ROOT/'proxy'/fold_name
shutil.copytree(proxy_source,proxy_destination,dirs_exist_ok=True)
import json
proxy_quality = json.loads((proxy_destination/'hr_proxy_validation_metrics.json').read_text())
detector_quality = json.loads((proxy_destination/'detector_validation_metrics.json').read_text())
print('Proxy validation MAE bpm:',proxy_quality['best_val_mae_bpm'])
print('Detector real validation metrics:',detector_quality['selected'])

# Verify the transferred input package before using its selected configuration.
import hashlib
manifest = json.loads((final_input/'source_manifest.json').read_text(encoding='utf-8'))
assert manifest['fold'] == FOLD
for relative, expected in manifest['files_sha256'].items():
    actual = hashlib.sha256((final_input/relative).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f'Input checksum mismatch: {relative}')
selected_source = final_input/'tuning'/'selected_config.json'
selected = json.loads(selected_source.read_text(encoding='utf-8'))
if selected['fold'] != FOLD:
    raise ValueError('Selected configuration belongs to a different fold')
tuning_destination = OUTPUT_ROOT/'tuning'/fold_name
shutil.copytree(final_input/'tuning', tuning_destination, dirs_exist_ok=True)
print('Selected configuration:', selected)
print('Training seeds:', SEEDS)
print('Exploratory fold 1 run: current validation HR accuracy is low; test remains closed.')


In [ ]:
from ppg_cvae.workflow import train_selected
from ppg_cvae.utils.io import save_json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    raise RuntimeError('This training job expects a GPU; run KRun with --gpu T4.')
plan = dict(fold=FOLD, seeds=list(SEEDS), beta=selected['beta'], lambda_hr=selected['lambda_hr'],
            tuning_source_job=manifest['source_job'], test_accessed=False,
            interpretation='exploratory; assess seed stability and validation HR quality before full cross validation')
save_json(plan, OUTPUT_ROOT/'cvae'/fold_name/'training_plan.json')
for seed in SEEDS:
    print(f'Starting fold {FOLD}, seed {seed}', flush=True)
    run_dir, history = train_selected(OUTPUT_ROOT, FOLD, seed, device=device)
    print('Saved checkpoint:', run_dir, flush=True)
    print('Epochs completed:', len(history), flush=True)
print('Completed fold 1 seeds 42/43/44. Inspect prior_validation_summary.csv for each seed.')
print('Other folds require their own prepared arrays, proxy, detector and tuning configuration.')
